ESERCIZIO

Implementa una funzione Python che collegata alla webcam verifiche l'attraversamento di una linea orizzontale posta a metà altezza del frame.
Aggiungi un contatore che distingua tra oggetti che si muovono verso l'alto e oggetti verso il basso.
Suggerimento: usa il segno della differenza tra la coordinata Y e la soglia della linea.

In [ ]:
"""
================================================================================
MONITORAGGIO DIREZIONALE (SU/GIU)
================================================================================
"""
import cv2
import datetime
from ultralytics import YOLO

# --- CONFIGURAZIONE ---
MODELLO = "yolov8m.pt"   # Modello Medium (COCO)
LINEA_Y = 350            # Altezza della linea orizzontale
FILE_LOG = "log_direzionale.csv"

# 1. SETUP INIZIALE
# Carichiamo il modello e configuriamo il tracker Kalman
model = YOLO(MODELLO) 
cap = cv2.VideoCapture(0)

# Variabili di stato
entrate = 0  # Movimento DOWN (verso il basso)
uscite = 0   # Movimento UP (verso l'alto)
id_contati = set()    # Per non contare due volte lo stesso passaggio
memoria_y = {}        # Dizionario: { ID_OGGETTO: ULTIMA_POSIZIONE_Y }

# Prepariamo il file CSV
with open(FILE_LOG, "a") as f:
    f.write("Timestamp,Direzione,Oggetto,Totale_Entrate,Totale_Uscite\n")

print(f"[INFO] Avvio sistema direzionale con {MODELLO}. Premi 'q' per uscire.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Ridimensioniamo per coerenza (1020x720 è un buon compromesso)
    frame = cv2.resize(frame, (1020, 720))

    # 2. TRACKING AI (ByteTrack gestisce frame skipping e Kalman)
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=0.5)

    if results[0].boxes.id is not None:
        # Estraiamo i dati
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()

        for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
            x1, y1, x2, y2 = map(int, box)
            cy = (y1 + y2) // 2  # Centro Verticale Attuale
            cx = (x1 + x2) // 2
            nome = model.names[cls_id]

            # 3. LOGICA DIREZIONALE
            # Se conosciamo dov'era questo ID nel frame precedente...
            if track_id in memoria_y:
                prev_y = memoria_y[track_id] # Posizione precedente

                # Controlla se ha attraversato la linea
                if track_id not in id_contati:
                    
                    # CASO 1: SCENDE (Era SOPRA < 350 e ora è SOTTO > 350)
                    # Nota: In OpenCV la Y cresce andando verso il basso (0 è in alto)
                    if prev_y < LINEA_Y and cy >= LINEA_Y:
                        entrate += 1
                        id_contati.add(track_id)
                        print(f"--> [ENTRATA] {nome} (ID {track_id})")
                        # Scrivi log
                        ora = datetime.datetime.now().strftime("%H:%M:%S")
                        with open(FILE_LOG, "a") as f:
                            f.write(f"{ora},ENTRATA,{nome},{entrate},{uscite}\n")

                    # CASO 2: SALE (Era SOTTO > 350 e ora è SOPRA < 350)
                    elif prev_y > LINEA_Y and cy <= LINEA_Y:
                        uscite += 1
                        id_contati.add(track_id)
                        print(f"<-- [USCITA] {nome} (ID {track_id})")
                        # Scrivi log
                        ora = datetime.datetime.now().strftime("%H:%M:%S")
                        with open(FILE_LOG, "a") as f:
                            f.write(f"{ora},USCITA,{nome},{entrate},{uscite}\n")

            # Aggiorniamo la memoria per il prossimo frame
            memoria_y[track_id] = cy

            # --- DISEGNO ---
            # Box Verde se contato, Rosso se ancora no
            colore = (0, 255, 0) if track_id in id_contati else (0, 0, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), colore, 2)
            cv2.putText(frame, f"ID:{track_id}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, colore, 2)
            cv2.circle(frame, (cx, cy), 4, (255, 255, 0), -1)

    # 4. INTERFACCIA UTENTE (HUD)
    # Linea di confine
    cv2.line(frame, (0, LINEA_Y), (1020, LINEA_Y), (0, 255, 255), 2)
    
    # Sfondo nero per i contatori
    cv2.rectangle(frame, (0, 0), (250, 80), (0,0,0), -1)
    cv2.putText(frame, f"ENTRATE (Giu): {entrate}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(frame, f"USCITE (Su):   {uscite}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Monitoraggio Direzionale", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()